# Query Decomposition with AWS

## 📖 Overview

This notebook demonstrates **Query Decomposition** for RAG using AWS services:
- **Amazon Bedrock Claude** for query decomposition and synthesis
- **AWS OpenSearch Serverless** for parallel retrieval
- **Amazon Bedrock Titan** for embeddings

### What is Query Decomposition?

Query Decomposition **breaks complex queries into simpler sub-questions**, retrieves information for each, and synthesizes a comprehensive answer.

**Problem with Complex Queries:**
```
Query: "Compare AWS Bedrock and SageMaker for ML workloads, 
        considering cost, ease of use, and scalability"
        
Traditional RAG → Single vector search may miss aspects
```

**Query Decomposition Solution:**
```
Main Query → Decompose into:
  1. "What is AWS Bedrock and its use cases?"
  2. "What is Amazon SageMaker and its use cases?"
  3. "What are the costs of Bedrock?"
  4. "What are the costs of SageMaker?"
  5. "How does Bedrock compare to SageMaker in ease of use?"
  6. "How do they compare in scalability?"
  
→ Retrieve for each sub-question in parallel
→ Synthesize comprehensive answer
```

### Why Decompose Queries?

**Benefits:**
- **Better Coverage**: Each aspect gets focused retrieval
- **Parallel Processing**: Sub-queries processed concurrently
- **Comprehensive Answers**: Addresses all parts of complex question
- **Clear Structure**: Organized by sub-topic

**Use Cases:**
- Multi-faceted questions
- Comparison queries
- "Explain A and B and how they relate"
- Questions with multiple parts

### When to Use

✅ **Good for:**
- Complex, multi-part questions
- Comparison queries
- Questions requiring multiple aspects
- When single query too broad
- Research-style questions

❌ **Not ideal for:**
- Simple factual questions
- Single-aspect queries
- When latency critical (multiple LLM calls)
- Very tight budgets
- Yes/no questions

### Architecture

```mermaid
graph TB
    A[Complex Query] --> B[Decompose into Sub-queries<br/>Claude]
    B --> C1[Sub-query 1]
    B --> C2[Sub-query 2]
    B --> C3[Sub-query 3]
    B --> C4[Sub-query 4]
    
    C1 --> D1[Retrieve Docs 1<br/>Parallel]
    C2 --> D2[Retrieve Docs 2<br/>Parallel]
    C3 --> D3[Retrieve Docs 3<br/>Parallel]
    C4 --> D4[Retrieve Docs 4<br/>Parallel]
    
    D1 --> E1[Answer 1]
    D2 --> E2[Answer 2]
    D3 --> E3[Answer 3]
    D4 --> E4[Answer 4]
    
    E1 --> F[Synthesize<br/>Comprehensive Answer]
    E2 --> F
    E3 --> F
    E4 --> F
    
    F --> G[Final Answer]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C1 fill:#e8f5e9
    style C2 fill:#e8f5e9
    style C3 fill:#e8f5e9
    style C4 fill:#e8f5e9
    style D1 fill:#f3e5f5
    style D2 fill:#f3e5f5
    style D3 fill:#f3e5f5
    style D4 fill:#f3e5f5
    style F fill:#ffe0b2
    style G fill:#c8e6c9
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict, Tuple
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'decomposition_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
DECOMPOSER_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'  # Quality for decomposition
ANSWERER_MODEL = 'us.anthropic.claude-haiku-3-20241022-v1:0'  # Fast for sub-answers
SYNTHESIZER_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'  # Quality for synthesis

# Decomposition Parameters
MAX_SUB_QUERIES = 6  # Maximum sub-queries to generate
RETRIEVAL_TOP_K = 3  # Documents per sub-query
PARALLEL_WORKERS = 4  # Concurrent sub-query processing

print(f"Configuration:")
print(f"  Decomposer: {DECOMPOSER_MODEL.split('.')[-1][:25]}")
print(f"  Answerer: {ANSWERER_MODEL.split('.')[-1][:25]}")
print(f"  Synthesizer: {SYNTHESIZER_MODEL.split('.')[-1][:25]}")
print(f"  Max sub-queries: {MAX_SUB_QUERIES}")
print(f"  Parallel workers: {PARALLEL_WORKERS}")

## 3️⃣ Initialize Services

In [ ]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
decomposer = BedrockLLM(AWS_REGION, DECOMPOSER_MODEL, temperature=0.7)
answerer = BedrockLLM(AWS_REGION, ANSWERER_MODEL, temperature=0.5)
synthesizer = BedrockLLM(AWS_REGION, SYNTHESIZER_MODEL, temperature=0.7)

print("✓ Services initialized")

## 4️⃣ Load Sample Documents

In [ ]:
sample_documents = [
    # AWS Bedrock
    "AWS Bedrock is a fully managed service providing access to foundation models from Anthropic, Amazon, Cohere, Meta, and others through a single API. It offers security, privacy, and responsible AI features.",
    "Bedrock pricing is based on input and output tokens. Claude Opus costs $15/$75 per million tokens, Sonnet $3/$15, and Haiku $0.25/$1.25. No infrastructure costs.",
    "Bedrock requires minimal setup - just API calls. No model training, deployment, or infrastructure management needed. Ideal for quick prototyping and production deployment.",
    "Bedrock automatically scales to handle varying workloads. It supports batch inference, streaming, and real-time generation. Global availability across AWS regions.",
    
    # Amazon SageMaker
    "Amazon SageMaker is a comprehensive ML platform for building, training, and deploying custom models. It provides notebooks, training jobs, model hosting, and MLOps tools.",
    "SageMaker pricing varies by instance type. Training on ml.p3.2xlarge costs ~$3/hour. Inference endpoints range from $0.05-$30/hour depending on instance size. Storage and data transfer are additional.",
    "SageMaker has a steeper learning curve than Bedrock. Requires ML expertise for model training, hyperparameter tuning, and deployment configuration. More flexibility but more complexity.",
    "SageMaker offers auto-scaling for inference endpoints, distributed training across multiple instances, and can handle massive datasets. Supports custom models and frameworks.",
    
    # RAG and Vector Search
    "RAG combines retrieval with generation, grounding LLM responses in external knowledge. Typical architecture: embed documents, retrieve similar content, generate answer with context.",
    "Vector embeddings capture semantic meaning in high-dimensional space. Similar content maps to nearby points. Enables semantic search beyond keyword matching.",
    "OpenSearch Serverless provides managed vector search with automatic scaling. Supports HNSW algorithm for efficient similarity search. No cluster management required.",
    
    # Cost Optimization
    "Cost optimization strategies include caching embeddings, using smaller models for simple tasks, batching requests, and implementing hybrid search to reduce vector computations.",
    "Bedrock's pay-per-token model can be more cost-effective for variable workloads. SageMaker endpoints have minimum costs but better for consistent high-volume usage.",
    
    # Use Cases
    "Bedrock excels at conversational AI, content generation, summarization, and Q&A without custom model training. Quick time-to-value for generative AI applications.",
    "SageMaker is ideal for custom ML models, computer vision, fraud detection, recommendation systems, and scenarios requiring fine-grained control over model architecture.",
]

# Index documents
opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

print("Indexing documents...")
documents = []
for i, text in enumerate(sample_documents):
    embedding = embedder.embed_text(text)
    documents.append({
        'text': text,
        'embedding': embedding,
        'metadata': {'doc_id': i}
    })

opensearch.index_documents(documents)
print(f"✓ Indexed {len(documents)} documents")

## 5️⃣ Query Decomposition

Break complex queries into focused sub-questions.

### Decomposition Strategy

We use Claude to:
1. **Analyze** the main question's structure
2. **Identify** key aspects and topics
3. **Generate** focused sub-questions
4. **Ensure** sub-questions cover the main question completely

**Good Sub-queries:**
- Focused on single aspect
- Answerable independently
- Together cover main question
- Avoid redundancy

In [ ]:
def decompose_query(query: str, max_sub_queries: int = 6) -> List[str]:
    """
    Decompose complex query into sub-questions
    
    Returns:
        List of sub-queries
    """
    decomposition_prompt = f"""
Break down the following complex question into simpler sub-questions.

Requirements:
- Each sub-question should focus on one specific aspect
- Sub-questions should be answerable independently
- Together, they should fully cover the main question
- Generate {max_sub_queries} or fewer sub-questions
- Number each sub-question (1., 2., etc.)

Main Question: {query}

Sub-questions:
"""
    
    try:
        response = decomposer.generate(decomposition_prompt, temperature=0.7)
        
        # Parse sub-questions
        lines = response.strip().split('\n')
        sub_queries = []
        
        for line in lines:
            line = line.strip()
            if line and line[0].isdigit():
                # Remove numbering
                query = line.split('. ', 1)[-1].split(') ', 1)[-1]
                sub_queries.append(query.strip())
        
        return sub_queries[:max_sub_queries]
    
    except Exception as e:
        print(f"Decomposition error: {e}")
        return [query]  # Fallback to original query

# Test decomposition
test_queries = [
    "Compare AWS Bedrock and SageMaker for ML workloads, considering cost, ease of use, and scalability",
    "How does RAG work and what are its main benefits?",
    "Explain vector search and how it's used in semantic search applications"
]

print("Query Decomposition Test\n" + "="*70 + "\n")
for i, query in enumerate(test_queries, 1):
    print(f"Main Query {i}:\n{query}\n")
    print("Sub-queries:")
    
    sub_queries = decompose_query(query, MAX_SUB_QUERIES)
    for j, sub_q in enumerate(sub_queries, 1):
        print(f"  {j}. {sub_q}")
    
    print(f"\nGenerated {len(sub_queries)} sub-queries\n")
    print("="*70 + "\n")

## 6️⃣ Parallel Sub-Query Processing

Process sub-queries concurrently for efficiency.

In [ ]:
def answer_sub_query(sub_query: str, top_k: int = 3) -> Dict:
    """
    Answer a single sub-query
    
    Returns:
        Dict with sub_query, answer, documents
    """
    try:
        # Retrieve relevant documents
        query_emb = embedder.embed_text(sub_query)
        results = opensearch.vector_search(query_emb, top_k=top_k)
        
        # Generate answer
        context_texts = [r['text'] for r in results]
        answer = answerer.generate_with_context(sub_query, context_texts)
        
        return {
            'sub_query': sub_query,
            'answer': answer,
            'documents': results,
            'success': True
        }
    except Exception as e:
        print(f"Error answering '{sub_query}': {e}")
        return {
            'sub_query': sub_query,
            'answer': f"Error: {str(e)}",
            'documents': [],
            'success': False
        }

def parallel_sub_query_answering(sub_queries: List[str], 
                                top_k: int = 3,
                                max_workers: int = 4) -> List[Dict]:
    """
    Answer multiple sub-queries in parallel
    
    Returns:
        List of answer dicts
    """
    print(f"Processing {len(sub_queries)} sub-queries in parallel...\n")
    
    results = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all sub-queries
        future_to_query = {
            executor.submit(answer_sub_query, sq, top_k): sq 
            for sq in sub_queries
        }
        
        # Collect results as they complete
        for future in as_completed(future_to_query):
            sub_query = future_to_query[future]
            try:
                result = future.result()
                results.append(result)
                status = "✓" if result['success'] else "✗"
                print(f"{status} Answered: {sub_query[:60]}...")
            except Exception as e:
                print(f"✗ Failed: {sub_query[:60]}... ({e})")
    
    print(f"\n✓ Completed {len(results)}/{len(sub_queries)} sub-queries\n")
    return results

# Test parallel processing
test_main_query = "Compare AWS Bedrock and SageMaker in terms of cost and ease of use"
print(f"Test Query: {test_main_query}\n" + "="*70 + "\n")

test_sub_queries = decompose_query(test_main_query, MAX_SUB_QUERIES)
print(f"Decomposed into {len(test_sub_queries)} sub-queries:\n")
for i, sq in enumerate(test_sub_queries, 1):
    print(f"{i}. {sq}")

print("\n" + "="*70 + "\n")

start_time = time.time()
sub_answers = parallel_sub_query_answering(test_sub_queries, RETRIEVAL_TOP_K, PARALLEL_WORKERS)
parallel_time = time.time() - start_time

print(f"Parallel processing time: {parallel_time:.2f}s")
print(f"Average per sub-query: {parallel_time/len(test_sub_queries):.2f}s\n")

print("Sub-answers:\n")
for i, sa in enumerate(sub_answers, 1):
    print(f"{i}. Q: {sa['sub_query']}")
    print(f"   A: {sa['answer'][:100]}...\n")

## 7️⃣ Answer Synthesis

Combine sub-answers into comprehensive response.

In [ ]:
def synthesize_answer(main_query: str, sub_answers: List[Dict]) -> str:
    """
    Synthesize final answer from sub-answers
    
    Returns:
        Comprehensive synthesized answer
    """
    # Build context from sub-answers
    context = "\n\n".join([
        f"Q: {sa['sub_query']}\nA: {sa['answer']}"
        for sa in sub_answers if sa['success']
    ])
    
    synthesis_prompt = f"""
Synthesize a comprehensive answer to the main question using the provided sub-question answers.

Main Question: {main_query}

Sub-question Answers:
{context}

Instructions:
- Create a well-structured, comprehensive answer
- Integrate information from all sub-answers
- Ensure logical flow and coherence
- Address all aspects of the main question
- Be concise but thorough

Comprehensive Answer:
"""
    
    try:
        final_answer = synthesizer.generate(synthesis_prompt, temperature=0.7)
        return final_answer
    except Exception as e:
        print(f"Synthesis error: {e}")
        # Fallback: concatenate sub-answers
        return "\n\n".join([sa['answer'] for sa in sub_answers if sa['success']])

# Test synthesis
print("Testing Answer Synthesis\n" + "="*70 + "\n")
print(f"Main Query: {test_main_query}\n")

synthesized = synthesize_answer(test_main_query, sub_answers)

print("Synthesized Answer:\n")
print(synthesized)
print("\n" + "="*70)

## 8️⃣ Complete Decomposition RAG Pipeline

In [ ]:
def decomposition_rag(query: str,
                     max_sub_queries: int = 6,
                     retrieval_top_k: int = 3,
                     parallel_workers: int = 4) -> Dict:
    """
    Complete query decomposition RAG pipeline
    
    Returns:
        Dict with answer, sub_queries, sub_answers, metadata
    """
    start_time = time.time()
    
    # Step 1: Decompose query
    print("Step 1: Decomposing query...")
    decompose_start = time.time()
    sub_queries = decompose_query(query, max_sub_queries)
    decompose_time = time.time() - decompose_start
    print(f"✓ Decomposed into {len(sub_queries)} sub-queries ({decompose_time:.2f}s)\n")
    
    # Step 2: Answer sub-queries in parallel
    print("Step 2: Answering sub-queries in parallel...")
    answer_start = time.time()
    sub_answers = parallel_sub_query_answering(sub_queries, retrieval_top_k, parallel_workers)
    answer_time = time.time() - answer_start
    print(f"Parallel answering time: {answer_time:.2f}s\n")
    
    # Step 3: Synthesize final answer
    print("Step 3: Synthesizing comprehensive answer...")
    synth_start = time.time()
    final_answer = synthesize_answer(query, sub_answers)
    synth_time = time.time() - synth_start
    print(f"✓ Synthesis complete ({synth_time:.2f}s)\n")
    
    total_time = time.time() - start_time
    print(f"Total time: {total_time:.2f}s")
    
    return {
        'answer': final_answer,
        'sub_queries': sub_queries,
        'sub_answers': sub_answers,
        'metadata': {
            'decompose_time': decompose_time,
            'answer_time': answer_time,
            'synthesis_time': synth_time,
            'total_time': total_time,
            'num_sub_queries': len(sub_queries),
            'successful_sub_answers': sum(1 for sa in sub_answers if sa['success'])
        }
    }

# Test complete pipeline
test_complex_queries = [
    "Compare AWS Bedrock and SageMaker for ML workloads, considering cost, ease of use, and scalability",
    "Explain how RAG works, its benefits, and how vector search enables it",
    "What are the main strategies for optimizing RAG systems in production?"
]

for i, query in enumerate(test_complex_queries, 1):
    print(f"\n{'='*70}")
    print(f"Complex Query {i}:\n{query}")
    print('='*70 + "\n")
    
    result = decomposition_rag(
        query,
        max_sub_queries=MAX_SUB_QUERIES,
        retrieval_top_k=RETRIEVAL_TOP_K,
        parallel_workers=PARALLEL_WORKERS
    )
    
    print(f"\n📊 Summary:")
    print(f"  Sub-queries generated: {result['metadata']['num_sub_queries']}")
    print(f"  Successful answers: {result['metadata']['successful_sub_answers']}")
    print(f"  Decomposition: {result['metadata']['decompose_time']:.2f}s")
    print(f"  Parallel answering: {result['metadata']['answer_time']:.2f}s")
    print(f"  Synthesis: {result['metadata']['synthesis_time']:.2f}s")
    print(f"  Total: {result['metadata']['total_time']:.2f}s")
    
    print(f"\n📝 Sub-queries:")
    for j, sq in enumerate(result['sub_queries'], 1):
        print(f"  {j}. {sq}")
    
    print(f"\n✅ Final Answer:\n{result['answer'][:300]}...")

## 9️⃣ Comparison: Decomposition vs Simple RAG

Compare quality and completeness.

In [ ]:
comparison_query = "Compare AWS Bedrock and SageMaker in terms of cost and use cases"

print("="*70)
print("DECOMPOSITION RAG")
print("="*70 + "\n")

# Decomposition approach
decomp_result = decomposition_rag(comparison_query, MAX_SUB_QUERIES, RETRIEVAL_TOP_K, PARALLEL_WORKERS)

print("\n" + "="*70)
print("SIMPLE RAG (Baseline)")
print("="*70 + "\n")

# Simple approach
print("Retrieving documents...")
simple_start = time.time()
query_emb = embedder.embed_text(comparison_query)
simple_results = opensearch.vector_search(query_emb, top_k=5)
simple_answer = synthesizer.generate_with_context(
    comparison_query,
    [r['text'] for r in simple_results]
)
simple_time = time.time() - simple_start
print(f"✓ Simple RAG completed in {simple_time:.2f}s\n")

# Comparison
print("="*70)
print("COMPARISON")
print("="*70)

print(f"\n⏱️  Time:")
print(f"  Decomposition RAG: {decomp_result['metadata']['total_time']:.2f}s")
print(f"  Simple RAG:        {simple_time:.2f}s")
print(f"  Overhead:          {decomp_result['metadata']['total_time'] - simple_time:.2f}s ({(decomp_result['metadata']['total_time']/simple_time - 1)*100:.0f}%)")

print(f"\n📚 Coverage:")
print(f"  Decomposition: {decomp_result['metadata']['num_sub_queries']} focused aspects")
print(f"  Simple: Single broad search")

print(f"\n📝 Answers:\n")
print(f"Decomposition RAG ({len(decomp_result['answer'].split())} words):\n{decomp_result['answer'][:400]}...\n")
print(f"Simple RAG ({len(simple_answer.split())} words):\n{simple_answer[:400]}...")

print(f"\n💡 Analysis:")
print(f"  - Decomposition provides more structured, comprehensive answer")
print(f"  - Each aspect (cost, use cases) addressed specifically")
print(f"  - Simple RAG faster but may miss nuances")
print(f"  - Trade-off: {(decomp_result['metadata']['total_time']/simple_time - 1)*100:.0f}% more time for better coverage")

## 🔟 Performance Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Analyze performance breakdown
print("Performance Analysis\n")
print("="*70 + "\n")

# Use last result for analysis
meta = decomp_result['metadata']

print(f"Time Breakdown:")
print(f"  1. Decomposition:    {meta['decompose_time']:>6.2f}s ({meta['decompose_time']/meta['total_time']*100:>5.1f}%)")
print(f"  2. Parallel Answer:  {meta['answer_time']:>6.2f}s ({meta['answer_time']/meta['total_time']*100:>5.1f}%)")
print(f"  3. Synthesis:        {meta['synthesis_time']:>6.2f}s ({meta['synthesis_time']/meta['total_time']*100:>5.1f}%)")
print(f"  {'─'*40}")
print(f"  Total:               {meta['total_time']:>6.2f}s")

# Visualize
stages = ['Decomposition', 'Parallel\nAnswering', 'Synthesis']
times = [meta['decompose_time'], meta['answer_time'], meta['synthesis_time']]
colors = ['#e1f5ff', '#f3e5f5', '#ffe0b2']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = ax1.bar(stages, times, color=colors, edgecolor='navy', linewidth=2)
ax1.set_ylabel('Time (seconds)', fontsize=12, fontweight='bold')
ax1.set_title('Decomposition RAG - Time Breakdown', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

for bar, time_val in zip(bars, times):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{time_val:.2f}s',
            ha='center', va='bottom', fontweight='bold')

# Pie chart
ax2.pie(times, labels=stages, autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Time Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Parallel efficiency
num_sub = meta['num_sub_queries']
sequential_estimate = meta['answer_time'] * num_sub / PARALLEL_WORKERS
speedup = sequential_estimate / meta['answer_time']

print(f"\n⚡ Parallel Processing Efficiency:")
print(f"  Sub-queries: {num_sub}")
print(f"  Workers: {PARALLEL_WORKERS}")
print(f"  Actual time: {meta['answer_time']:.2f}s")
print(f"  Sequential estimate: {sequential_estimate:.2f}s")
print(f"  Speedup: {speedup:.1f}x")

# Cost estimation
decompose_cost = 0.015  # Opus for decomposition
answer_cost = num_sub * 0.00025  # Haiku per sub-query
synthesis_cost = 0.015  # Opus for synthesis
total_cost = decompose_cost + answer_cost + synthesis_cost

simple_cost = 0.015  # Opus for simple RAG

print(f"\n💰 Cost Estimate (per query):")
print(f"  Decomposition:       ${decompose_cost:.5f}")
print(f"  Sub-answers ({num_sub}):     ${answer_cost:.5f}")
print(f"  Synthesis:           ${synthesis_cost:.5f}")
print(f"  {'─'*40}")
print(f"  Total:               ${total_cost:.5f}")
print(f"\n  Simple RAG:          ${simple_cost:.5f}")
print(f"  Additional cost:     ${total_cost - simple_cost:.5f} ({(total_cost/simple_cost - 1)*100:+.1f}%)")

print(f"\n🎯 Trade-off Summary:")
print(f"  Time: +{(meta['total_time']/simple_time - 1)*100:.0f}% vs Simple RAG")
print(f"  Cost: +{(total_cost/simple_cost - 1)*100:.0f}% vs Simple RAG")
print(f"  Benefit: Comprehensive coverage of complex questions")
print(f"  Best for: Multi-faceted queries requiring thorough analysis")

## 1️⃣1️⃣ Summary & Key Takeaways

### What We Built

✅ Query decomposition with LLM
✅ Parallel sub-query processing
✅ Answer synthesis from sub-answers
✅ Complete decomposition RAG pipeline
✅ Performance analysis and comparison

### Performance Characteristics

- **Latency**: 2-3x slower than simple RAG (but parallel helps)
- **Cost**: ~2x simple RAG (multiple LLM calls)
- **Quality**: Better coverage and structure for complex queries
- **Parallelization**: 2-4x speedup with concurrent processing

### When to Use Query Decomposition

**Use Decomposition when:**
- Queries have multiple distinct aspects
- Comparison questions ("A vs B")
- "Explain X, Y, and Z" questions
- Research-style queries
- Need comprehensive, structured answers

**Skip Decomposition when:**
- Simple factual questions
- Single-aspect queries
- Latency critical
- Very tight budget
- Yes/no questions

### Key Insights

1. **Focused Retrieval**: Each sub-query retrieves specifically relevant docs
2. **Parallel Wins**: Concurrent processing reduces wall-clock time significantly
3. **Better Structure**: Sub-answers provide organized information
4. **Comprehensive**: Less likely to miss important aspects
5. **Cost vs Quality**: 2x cost for significantly better complex query handling

### Decomposition Strategies

**1. Aspect-based (This Notebook)**
- Break by key aspects (cost, features, etc.)
- Good for comparison and analysis

**2. Sequential**
- Each sub-query builds on previous
- Good for step-by-step explanations

**3. Hierarchical**
- Main topics → sub-topics
- Good for broad questions

**4. Entity-based**
- One sub-query per entity
- Good for multi-entity questions

### Best Practices

1. **Limit Sub-queries**: 3-6 is optimal (balance coverage vs overhead)
2. **Use Parallel Processing**: Dramatically reduces latency
3. **Model Selection**: Fast model (Haiku) for sub-answers, quality (Opus) for synthesis
4. **Error Handling**: Continue if some sub-queries fail
5. **Cache Decompositions**: Similar queries often have similar decompositions

### Optimization Techniques

**Async Processing:**
```python
import asyncio
# Process all sub-queries truly concurrently
async def async_answer_sub_queries(sub_queries):
    tasks = [answer_sub_query_async(sq) for sq in sub_queries]
    return await asyncio.gather(*tasks)
```

**Adaptive Depth:**
```python
# Fewer sub-queries for simple questions
if query_complexity < 0.5:
    max_sub_queries = 3
else:
    max_sub_queries = 6
```

**Smart Synthesis:**
```python
# Use cheaper model if sub-answers already comprehensive
if sum(len(sa['answer']) for sa in sub_answers) < 500:
    synthesizer = haiku_model  # Cheaper
```

### Limitations

1. **Higher Latency**: Even with parallelization, slower than simple RAG
2. **Higher Cost**: Multiple LLM calls (decompose + N answers + synthesize)
3. **Decomposition Quality**: Bad decomposition = bad results
4. **Overkill for Simple**: Wastes resources on straightforward questions
5. **Synthesis Complexity**: Combining diverse sub-answers challenging

### Combining with Other Patterns

**Decomposition + Adaptive:**
- Classify query first
- Use decomposition only for complex queries
- Best of both worlds

**Decomposition + HyDE:**
- Use HyDE for each sub-query
- Better retrieval per aspect

**Decomposition + Reranking:**
- Rerank results for each sub-query
- Highest precision

### Advanced Topics

- **Iterative Decomposition**: Refine sub-queries based on initial results
- **Dependency-aware**: Some sub-queries depend on others
- **Hierarchical Synthesis**: Multi-level answer assembly
- **Learned Decomposition**: Train model to generate optimal sub-queries

### Metrics to Track

1. **Decomposition Quality**: Are sub-queries well-formed?
2. **Coverage**: Do sub-queries cover main question?
3. **Redundancy**: Are sub-queries overlapping?
4. **Success Rate**: % sub-queries answered successfully
5. **Synthesis Quality**: Is final answer coherent?
6. **User Satisfaction**: Better than simple RAG?

### Next Steps

- **Async implementation**: True concurrent processing
- **Streaming synthesis**: Stream answer as sub-answers complete
- **Adaptive decomposition**: Adjust number based on query
- **Dependency detection**: Order sub-queries optimally
- **Fine-tune decomposer**: Train on domain-specific queries

## 🧹 Cleanup

In [ ]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")